**1. DATA CLEANSING**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv('student_depression_dataset.csv')
df.info()

memeriksa nilai kosong

In [ ]:
print("Missing Values per Column:", df.isnull().sum())

menghapus data duplikat

In [ ]:
df.drop_duplicates()
df.drop(columns='id')

penanganan data inkonsisten

In [ ]:
cols_yes_no = ['Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']
for col in cols_yes_no:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].replace({'yes': 'Yes', 'no': 'No'})

print("Penanganan Data Inkonsisten (setelah koreksi):", df[cols_yes_no].drop_duplicates())


mengoreksi nilai tidak valid

In [ ]:
if 'Age' in df.columns:
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df = df[(df['Age'] >= 15) & (df['Age'] <= 50)]
    print("Jumlah data setelah membuang nilai usia tidak valid:", df.shape[0])

mengganti nilai tidak valid

In [ ]:
targets = ['??', 'N/A', 'n/a', 'unknown', 'Unknown']
df.replace(targets, np.nan, inplace=True)
print(df.isin(targets).sum())

memeriksa outlier

In [ ]:
for col in df.select_dtypes(include=[np.number]).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = df.shape[0]
    df = df[(df[col] >= lower) & (df[col] <= upper)]
    after = df.shape[0]
    print(f"Outlier dibuang dari kolom '{col}': {before - after}")

print("Jumlah data setelah buang outlier:", df.shape[0])

**2. DATA TRANSFORMATION**

Label Encoding:

mengubah kolom dengan tipe data Kategorikal fitur Biner: 'Gender', 'Have you ever had suicidal thoughts?', dan 'Family History of Mental Illness'

df['Gender'] = df['Gender'].map(
    {'Male': 0,
     'Female': 1})
df['Have you ever had suicidal thoughts ?'] = df['Have you ever had suicidal thoughts ?'].map(
    {'Yes': 1,
     'No': 0})
df['Family History of Mental Illness'] = df['Family History of Mental Illness'].map(
    {'Yes': 1,
     'No': 0})

In [ ]:
df['Gender'] = df['Gender'].map(
    {'Male': 0,
     'Female': 1})
df['Have you ever had suicidal thoughts ?'] = df['Have you ever had suicidal thoughts ?'].map(
    {'Yes': 1,
     'No': 0})
df['Family History of Mental Illness'] = df['Family History of Mental Illness'].map(
    {'Yes': 1,
     'No': 0})

In [ ]:
print("Gender: ")
print(df['Gender'].head())

print("Have you ever had suicidal thoughts?: ")
print(df['Have you ever had suicidal thoughts ?'].head())

print("Family History of Mental Illness: ")
print(df['Family History of Mental Illness'].head())

One-Hot Encoding

untuk mengubah kolom dengan Kategorikal Nominal yaitu 'City', 'Profession', 'Degree', 'Sleep Duration', dan 'Dietary Habits'

In [ ]:
df = pd.get_dummies(df, columns=['City', 'Profession', 'Degree', 'Sleep Duration', 'Dietary Habits'])
df.head()

Normalization dengan MinMaxScaler

untuk mengubah nilai fitur (Depression) ke dalam rentang 0 - 1

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['Depression'] = scaler.fit_transform(df[['Depression']])
df.head()

Handling Missing Values

dengan mengecek Missing Value dan mengisi missing value jika ada

In [ ]:
df.isnull().sum()

**3. DATA REDUCTION**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

In [ ]:
df.nunique()

In [ ]:
numeric_cols = [ 'Gender', 'Age', 'Academic Pressure', 'Work Pressure', 'CGPA']
df_pca = df[numeric_cols]

In [ ]:
print(df_pca.dtypes)
print(df_pca.apply(lambda x: x.unique()))

In [ ]:
df['Financial Stress'] = pd.to_numeric(df['Financial Stress'], errors='coerce')

In [ ]:
print(df['Financial Stress'].isnull().sum())

In [ ]:
print(df['Financial Stress'].isnull().sum())

In [ ]:
df['Financial Stress'] = df['Financial Stress'].replace('?', np.nan)
df['Financial Stress'] = pd.to_numeric(df['Financial Stress'])

In [ ]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
print("Kolom numerik yang siap diproses:", list(numerical_cols))

In [ ]:
scaler = StandardScaler()
scaler.fit_transform(df_pca)

In [ ]:
pca = PCA(n_components=2)

principalComponents = pca.fit_transform(df_pca)
principalDf = pd.DataFrame(data = principalComponents, columns = ['PC1', 'PC2'])

In [ ]:
df[numeric_cols].head(5)

In [ ]:
principalDf

In [ ]:
print("Variansi per komponen:", pca.explained_variance_ratio_)
print("Total variansi terjaga:", sum(pca.explained_variance_ratio_))

In [ ]:
df_sample = df.sample(frac=0.8, random_state=42)

In [ ]:
df_sample.info()

In [ ]:
df_sample.head()

**4. DATA MODELLING**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
X = df.drop(['Depression'], axis=1)
y = df['Depression']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print("Hasil Prediksi (y_pred):")
print(y_pred)

**5. DATA EVALUATION**

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [ ]:
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("R-squared (R2):", r2)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Nilai Aktual Tingkat Depresi")
plt.ylabel("Prediksi Tingkat Depresi")
plt.title("Aktual vs. Prediksi Tingkat Depresi")
plt.grid(True)
plt.show()

In [ ]:
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True)
plt.xlabel("Residuals")
plt.ylabel("Frekuensi")
plt.title("Distribusi Residuals")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.xlabel("Fitted Values (Prediksi)")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values")
plt.axhline(y=0, color='r', linestyle='--')
plt.grid(True)
plt.show()

**6. DATA DEPLOYMENT**

In [ ]:
from sklearn.linear_model import LinearRegression
import joblib

model = LinearRegression()
model.fit(X_train, y_train)  

joblib.dump(model, 'linear_regression_model.pkl')         
joblib.dump(scaler, 'minmax_scaler.pkl')                  
joblib.dump(X.columns.tolist(), 'X_train_cols.pkl')  

In [ ]:
!ngrok config add-authtoken  2wwBRB4AGdfGC4Ys1FTD08ui55j_6JXQz1CZCH3houhHre2q6

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib 

# Fungsi untuk memuat model, scaler, dan daftar kolom
@st.cache_resource
def load_artifacts():
    try:
        model = joblib.load('linear_regression_model.pkl')
        scaler = joblib.load('minmax_scaler.pkl')
        X_train_cols = joblib.load('X_train_cols.pkl')
        return model, scaler, X_train_cols
    except FileNotFoundError:
        st.error("File model, scaler, atau kolom tidak ditemukan. Pastikan Anda telah mengunggah file-file ini.")
        return None, None, None

model, scaler, X_train_cols = load_artifacts()

if model is not None and scaler is not None and X_train_cols is not None:
    st.title("Dashboard Prediksi Tingkat Depresi Mahasiswa")

    st.sidebar.header("Input Data Mahasiswa")

    # Input dari pengguna di sidebar
    gender = st.sidebar.selectbox("Gender", ["Male", "Female"])
    age = st.sidebar.slider("Age", 15, 50, 20)
    academic_pressure = st.sidebar.slider("Academic Pressure", 0, 100, 50)
    work_pressure = st.sidebar.slider("Work Pressure", 0, 100, 50)
    cgpa = st.sidebar.slider("IPK", 0.0, 4.0, 3.0)
    financial_stress = st.sidebar.slider("Financial Stress", 0, 100, 50)
    suicidal_thoughts = st.sidebar.selectbox("Have you ever had suicidal thoughts?", ["Yes", "No"])
    family_history = st.sidebar.selectbox("Family History of Mental Illness", ["Yes", "No"])
    city = st.sidebar.selectbox("City", ["Bandung", "Jakarta", "Others"]) 
    profession = st.sidebar.selectbox("Profession", ["Student", "Employed", "Others"]) 
    degree = st.sidebar.selectbox("Degree", ["Diploma", "Sarjana", "Master"])
    sleep_duration = st.sidebar.selectbox("Sleep Duration", ["1 - 3 Hours", "4 - 7 hours", " > 8 hours"]) 
    dietary_habits = st.sidebar.selectbox("Dietary Habits", ["Healthy", "Unhealthy"]) 

    # Pra-pemrosesan data input
    input_data = {
        'Gender': gender,
        'Age': age,
        'Academic Pressure': academic_pressure,
        'Work Pressure': work_pressure,
        'CGPA': cgpa,
        'Financial Stress': financial_stress,
        'Have you ever had suicidal thoughts ?': suicidal_thoughts,
        'Family History of Mental Illness': family_history,
        'City': city,
        'Profession': profession,
        'Degree': degree,
        'Sleep Duration': sleep_duration,
        'Dietary Habits': dietary_habits
    }

    input_df = pd.DataFrame([input_data])

    # Transformasi 
    input_df['Gender'] = input_df['Gender'].map({'Male': 0, 'Female': 1})
    input_df['Have you ever had suicidal thoughts ?'] = input_df['Have you ever had suicidal thoughts ?'].map({'Yes': 1, 'No': 0})
    input_df['Family History of Mental Illness'] = input_df['Family History of Mental Illness'].map({'Yes': 1, 'No': 0})

    # One-Hot Encoding
    input_df = pd.get_dummies(input_df, columns=['City', 'Profession', 'Degree', 'Sleep Duration', 'Dietary Habits'])

    # Pastikan semua kolom yang ada saat training ada di input_df
    for col in X_train_cols:
        if col not in input_df.columns:
            input_df[col] = 0

    input_df = input_df[X_train_cols] 


    # Prediksi
    if st.sidebar.button("Prediksi"):
        prediction = model.predict(input_df)
        st.subheader("Hasil Prediksi Tingkat Depresi:")
        predicted_depression_level = scaler.inverse_transform(prediction.reshape(-1, 1))
        st.write(f"Tingkat Depresi yang Diprediksi: {prediction[0][0]:.2f}")

        clipped_prediction = np.clip(prediction[0], 0, 10)
        result = prediction[0]
        result = max(0, min(result, 10))
        st.write(f"Tingkat Depresi yang Diprediksi: {clipped_prediction:.2f}")

        st.markdown("---")
        st.write("Dashboard ini memberikan prediksi tingkat depresi berdasarkan data input.")

else:
    st.warning("Tidak dapat memuat file model, scaler, atau kolom. Pastikan file-file tersebut ada.")

In [ ]:
from pyngrok import ngrok
import threading
import time
import os

In [ ]:
def run_streamlit():
    os.system("streamlit run tubesdatmin.py --server.port 8501 > /dev/null 2>&1")

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

time.sleep(10) 
public_url = ngrok.connect(addr=8501)

print(f"Streamlit app is live at:?{public_url}")